# Deutsch-Jozsa Algorithm

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import numpy as np

## Set a random seed so that the same oracle is generated each time

In [ ]:
np.random.seed(1992)

## Create a circuit that acts as an oracle for the Deutsch-Jozsa problem

In [ ]:
def create_oracle(num_qubits):
    """
    Create a random oracle circuit for the Deutsch-Jozsa algorithm.

    The oracle implements a Boolean function that is guaranteed to be
    either constant or balanced.
    """
    # The circuit uses num_qubits input qubits and 1 output qubit
    qc = QuantumCircuit(num_qubits + 1)

    # With 50% probability, flip the output qubit first
    # This changes the constant output value or swaps 0/1 labels
    if np.random.randint(0, 2):
        qc.x(num_qubits)

    # With 50% probability, return here to make the oracle constant
    if np.random.randint(0, 2):
        return qc

    # Otherwise, create a balanced oracle by selecting exactly half
    # of all possible input states for which the output will be flipped
    on_states = np.random.choice(
        range(2**num_qubits),   # all possible input basis states
        2**num_qubits // 2,     # choose exactly half of them
        replace=False,          # do not select the same state twice
    )

    def add_cx(qc, bit_string):
        """
        Apply X gates to input qubits corresponding to 1s in bit_string.

        This maps the chosen computational basis state to the all-ones state
        so that a multi-controlled X gate can target only that state.
        """
        for qubit, bit in enumerate(reversed(bit_string)):
            if bit == "1":
                qc.x(qubit)
        return qc

    # For each selected input state, flip the output qubit
    for state in on_states:
        # Convert the integer state into its binary representation
        qc = add_cx(qc, f"{state:0b}")

        # Flip the output qubit if all input qubits are in |1>
        qc.mcx(list(range(num_qubits)), num_qubits)

        # Undo the X gates so the basis returns to its original labeling
        qc = add_cx(qc, f"{state:0b}")

    return qc

In [ ]:
# Create a 3-input oracle

oracle_circuit = create_oracle(3)

In [ ]:
# Draw the oracle circuit
oracle_circuit.draw('mpl')

## Build the Deutsch-Jozsa algorithm circuit using the oracle

In [ ]:
n = oracle_circuit.num_qubits - 1
qc = QuantumCircuit(n + 1, n)

# Prepare the ancilla qubit in |1>
qc.x(n)

# Apply Hadamard gates to all qubits
# Input qubits become a uniform superposition
# The ancilla becomes |-> = (|0> - |1>) / sqrt(2)
qc.h(range(n + 1))
qc.barrier()

# Apply the oracle
qc.compose(oracle_circuit, inplace=True)
qc.barrier()

# Apply Hadamard gates again to the input qubits
qc.h(range(n))
qc.barrier()

# Measure the input qubits
qc.measure(range(n), range(n))

In [ ]:
# Draw the full Deutsch-Jozsa circuit
qc.draw('mpl')

## Simulate the circuit

In [ ]:
simulator = AerSimulator()
result = simulator.run(qc, shots=1, memory=True).result()
measurements = result.get_memory()

- If the result contains any '1', the function is balanced
- If the result is all zeros, the function is constant

In [ ]:
is_balanced = "1" in measurements[0]

print(f'{is_balanced=}')